In [2]:
#Import all the necessary modules
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
import seaborn as sns
import random
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
# calculate accuracy measures and confusion matrix
from sklearn import metrics
num_bins = 10


In [3]:
dbody=pd.read_csv("data/diabetes.csv")
dmind=pd.read_csv("data/Mental_Health_Dataset.csv")
print(f"Body Columns are:{list(dbody.columns)}")
print(f"Mind Columns are:{list(dmind.columns)}")




Body Columns are:['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
Mind Columns are:['Timestamp', 'Gender', 'Country', 'Occupation', 'self_employed', 'family_history', 'treatment', 'Days_Indoors', 'Growing_Stress', 'Changes_Habits', 'Mental_Health_History', 'Mood_Swings', 'Coping_Struggles', 'Work_Interest', 'Social_Weakness', 'mental_health_interview', 'care_options']


In [4]:
print(dbody)

     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0              6      148             72             35        0  33.6   
1              1       85             66             29        0  26.6   
2              8      183             64              0        0  23.3   
3              1       89             66             23       94  28.1   
4              0      137             40             35      168  43.1   
..           ...      ...            ...            ...      ...   ...   
763           10      101             76             48      180  32.9   
764            2      122             70             27        0  36.8   
765            5      121             72             23      112  26.2   
766            1      126             60              0        0  30.1   
767            1       93             70             31        0  30.4   

     DiabetesPedigreeFunction  Age  Outcome  
0                       0.627   50        1  
1                  

In [5]:
print(dmind)

              Timestamp  Gender        Country Occupation self_employed  \
0       8/27/2014 11:29  Female  United States  Corporate           NaN   
1       8/27/2014 11:31  Female  United States  Corporate           NaN   
2       8/27/2014 11:32  Female  United States  Corporate           NaN   
3       8/27/2014 11:37  Female  United States  Corporate            No   
4       8/27/2014 11:43  Female  United States  Corporate            No   
...                 ...     ...            ...        ...           ...   
292359  7/27/2015 23:25    Male  United States   Business           Yes   
292360   8/17/2015 9:38    Male   South Africa   Business            No   
292361  8/25/2015 19:59    Male  United States   Business            No   
292362   9/26/2015 1:07    Male  United States   Business            No   
292363   2/1/2016 23:04    Male  United States   Business            No   

       family_history treatment Days_Indoors Growing_Stress Changes_Habits  \
0                  No

In [6]:
dbody.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [7]:
print(dmind.dtypes)

Timestamp                  object
Gender                     object
Country                    object
Occupation                 object
self_employed              object
family_history             object
treatment                  object
Days_Indoors               object
Growing_Stress             object
Changes_Habits             object
Mental_Health_History      object
Mood_Swings                object
Coping_Struggles           object
Work_Interest              object
Social_Weakness            object
mental_health_interview    object
care_options               object
dtype: object


In [8]:
dmind.drop(['Country','Timestamp'], axis=1, inplace=True)

In [9]:
summary = {}

for col in dmind.columns:
    summary[col] = {
        "unique_count": dmind[col].nunique(),
        "top_values": dmind[col].value_counts(dropna=False).head(5).to_dict()
    }

summary

{'Gender': {'unique_count': 2,
  'top_values': {'Male': 239850, 'Female': 52514}},
 'Occupation': {'unique_count': 5,
  'top_values': {'Housewife': 66351,
   'Student': 61794,
   'Corporate': 61229,
   'Others': 52841,
   'Business': 50149}},
 'self_employed': {'unique_count': 2,
  'top_values': {'No': 257994, 'Yes': 29168, nan: 5202}},
 'family_history': {'unique_count': 2,
  'top_values': {'No': 176832, 'Yes': 115532}},
 'treatment': {'unique_count': 2, 'top_values': {'Yes': 147606, 'No': 144758}},
 'Days_Indoors': {'unique_count': 5,
  'top_values': {'1-14 days': 63548,
   '31-60 days': 60705,
   'Go out Every day': 58366,
   'More than 2 months': 55916,
   '15-30 days': 53829}},
 'Growing_Stress': {'unique_count': 3,
  'top_values': {'Maybe': 99985, 'Yes': 99653, 'No': 92726}},
 'Changes_Habits': {'unique_count': 3,
  'top_values': {'Yes': 109523, 'Maybe': 95166, 'No': 87675}},
 'Mental_Health_History': {'unique_count': 3,
  'top_values': {'No': 104018, 'Maybe': 95378, 'Yes': 92968

In [10]:
for col in dmind.columns:
    print("\n", col)
    print(dmind[col].unique())


 Gender
['Female' 'Male']

 Occupation
['Corporate' 'Student' 'Business' 'Housewife' 'Others']

 self_employed
[nan 'No' 'Yes']

 family_history
['No' 'Yes']

 treatment
['Yes' 'No']

 Days_Indoors
['1-14 days' 'Go out Every day' 'More than 2 months' '15-30 days'
 '31-60 days']

 Growing_Stress
['Yes' 'No' 'Maybe']

 Changes_Habits
['No' 'Yes' 'Maybe']

 Mental_Health_History
['Yes' 'No' 'Maybe']

 Mood_Swings
['Medium' 'Low' 'High']

 Coping_Struggles
['No' 'Yes']

 Work_Interest
['No' 'Maybe' 'Yes']

 Social_Weakness
['Yes' 'No' 'Maybe']

 mental_health_interview
['No' 'Maybe' 'Yes']

 care_options
['Not sure' 'No' 'Yes']


In [11]:
print(dmind['Occupation'].unique())

['Corporate' 'Student' 'Business' 'Housewife' 'Others']


In [12]:
dmind = dmind.applymap(lambda x: str(x).strip().lower() if isinstance(x, str) else x)

In [13]:
print(dmind['treatment'].unique())

['yes' 'no']


In [14]:
def simple_map(x):
    if x == "yes":
        return 1
    elif x == "no":
        return 0
    else:
        return 0.5

In [15]:
yes_no_cols = [
    'self_employed',
    'family_history',
    'treatment',
    'Growing_Stress',
    'Changes_Habits',
    'Mental_Health_History',
    'Coping_Struggles',
    'Work_Interest',
    'Social_Weakness',
    'mental_health_interview',
    'care_options',
    
]
for col in yes_no_cols:
    dmind[col] = dmind[col].apply(simple_map)

In [16]:
print(dmind['treatment'].unique())

[1 0]


In [17]:
print(dmind['Occupation'].unique())

['corporate' 'student' 'business' 'housewife' 'others']


In [18]:
work_map = {'corporate': 0, 'student': 1, 'business': 2, 'housewife': 3, 'others': 4}
dmind['Occupation'] = dmind['Occupation'].str.strip().str.lower().map(work_map)

In [19]:
print(dmind['Days_Indoors'].unique())

['1-14 days' 'go out every day' 'more than 2 months' '15-30 days'
 '31-60 days']


In [20]:
emp_map = {
    '1-14 days': 0, 'go out every day': 1, 'more than 2 months': 2, 
    '15-30 days': 3, '31-60 days': 4
}
dmind['Days_Indoors'] = dmind['Days_Indoors'].str.strip().str.lower().map(emp_map)

In [21]:
leave_map = {'medium': 0, 'low': 1, 'high': 2}
dmind['Mood_Swings'] = dmind['Mood_Swings'].str.strip().str.lower().map(leave_map)

In [22]:
gender_map = {'male': 0, 'female': 1}
dmind['Gender'] = dmind['Gender'].str.strip().str.lower().map(gender_map)

In [23]:
dmind.isnull().sum()

Gender                     0
Occupation                 0
self_employed              0
family_history             0
treatment                  0
Days_Indoors               0
Growing_Stress             0
Changes_Habits             0
Mental_Health_History      0
Mood_Swings                0
Coping_Struggles           0
Work_Interest              0
Social_Weakness            0
mental_health_interview    0
care_options               0
dtype: int64

In [24]:
dmind

,Gender,Occupation,self_employed,family_history,treatment,Days_Indoors,Growing_Stress,Changes_Habits,Mental_Health_History,Mood_Swings,Coping_Struggles,Work_Interest,Social_Weakness,mental_health_interview,care_options
0,1,0,0.5,0,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,0.5
1,1,0,0.5,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,0.0
2,1,0,0.5,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,1.0
3,1,0,0.0,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.5,1.0
4,1,0,0.0,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292359,0,2,1.0,1,1,3,0.0,0.5,0.0,1,1,0.0,0.5,0.5,0.5
292360,0,2,0.0,1,1,3,0.0,0.5,0.0,1,1,0.0,0.5,0.0,1.0
292361,0,2,0.0,1,0,3,0.0,0.5,0.0,1,1,0.0,0.5,0.0,0.0
292362,0,2,0.0,1,1,3,0.0,0.5,0.0,1,1,0.0,0.5,0.0,1.0


In [25]:
dmind.dtypes

Gender                       int64
Occupation                   int64
self_employed              float64
family_history               int64
treatment                    int64
Days_Indoors                 int64
Growing_Stress             float64
Changes_Habits             float64
Mental_Health_History      float64
Mood_Swings                  int64
Coping_Struggles             int64
Work_Interest              float64
Social_Weakness            float64
mental_health_interview    float64
care_options               float64
dtype: object

In [26]:
dmind

,Gender,Occupation,self_employed,family_history,treatment,Days_Indoors,Growing_Stress,Changes_Habits,Mental_Health_History,Mood_Swings,Coping_Struggles,Work_Interest,Social_Weakness,mental_health_interview,care_options
0,1,0,0.5,0,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,0.5
1,1,0,0.5,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,0.0
2,1,0,0.5,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,1.0
3,1,0,0.0,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.5,1.0
4,1,0,0.0,1,1,0,1.0,0.0,1.0,0,0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292359,0,2,1.0,1,1,3,0.0,0.5,0.0,1,1,0.0,0.5,0.5,0.5
292360,0,2,0.0,1,1,3,0.0,0.5,0.0,1,1,0.0,0.5,0.0,1.0
292361,0,2,0.0,1,0,3,0.0,0.5,0.0,1,1,0.0,0.5,0.0,0.0
292362,0,2,0.0,1,1,3,0.0,0.5,0.0,1,1,0.0,0.5,0.0,1.0


In [27]:
print(dmind.corr()['treatment'].sort_values(ascending=False))

treatment                  1.000000
family_history             0.366781
care_options               0.253428
Gender                     0.177203
self_employed              0.038028
Coping_Struggles           0.009985
Growing_Stress             0.007473
Occupation                 0.005291
Mood_Swings                0.003828
Work_Interest              0.003188
Days_Indoors               0.002936
Mental_Health_History      0.001492
Social_Weakness           -0.000821
Changes_Habits            -0.002203
mental_health_interview   -0.067147
Name: treatment, dtype: float64


In [28]:
print(dmind.dtypes)
print(dmind.head(3))

Gender                       int64
Occupation                   int64
self_employed              float64
family_history               int64
treatment                    int64
Days_Indoors                 int64
Growing_Stress             float64
Changes_Habits             float64
Mental_Health_History      float64
Mood_Swings                  int64
Coping_Struggles             int64
Work_Interest              float64
Social_Weakness            float64
mental_health_interview    float64
care_options               float64
dtype: object
   Gender  Occupation  self_employed  family_history  treatment  Days_Indoors  \
0       1           0            0.5               0          1             0   
1       1           0            0.5               1          1             0   
2       1           0            0.5               1          1             0   

   Growing_Stress  Changes_Habits  Mental_Health_History  Mood_Swings  \
0             1.0             0.0                    1.0       

In [29]:
print(dmind['treatment'].value_counts())

treatment
1    147606
0    144758
Name: count, dtype: int64


In [30]:
dmind['treatment'] = dmind['treatment'].astype(int)

In [31]:
dmind = dmind.fillna(0)
dbody=dbody.fillna(0)

In [32]:
Xbody = dbody.drop('Outcome', axis=1) # Copying all the predictor variables into X dataframe. 'Final_grade' is dropped as it is dependent variable
Ybody = dbody['Outcome']# Copy the 'Final_grade' column alone into the y dataframe. This is the dependent variable 
seed=25
#now we break the X and y dataframes into training set and test set. For this we will use
#Sklearn package's data splitting function which is based on random function.
Xbody_train, Xbody_test, Ybody_train, Ybody_test = train_test_split(Xbody, Ybody, test_size=0.2, random_state=25,stratify=Ybody)# Splitting X and y into training and test set in 80:20 ratio

In [33]:
# Import the Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest model with a fixed random state for reproducibility
rf_model_body = RandomForestClassifier(class_weight='balanced',random_state=25)

# Train the model on the training data
rf_model_body.fit(Xbody_train, Ybody_train)

# Predict the target values for the test data
rf_pred_body = rf_model_body.predict(Xbody_test)


In [34]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(Ybody_test, rf_pred_body))# Calculating the accuracy score of the model
print(classification_report(Ybody_test, rf_pred_body))
print("CONFUSION MATRIX:",metrics.confusion_matrix(Ybody_test, rf_pred_body))#Printing the confusion matrix to evaluate prediction results

Accuracy: 0.7337662337662337
              precision    recall  f1-score   support

           0       0.76      0.86      0.81       100
           1       0.66      0.50      0.57        54

    accuracy                           0.73       154
   macro avg       0.71      0.68      0.69       154
weighted avg       0.73      0.73      0.72       154

CONFUSION MATRIX: [[86 14]
 [27 27]]


In [35]:
Xmind = dmind.drop('treatment', axis=1) # Copying all the predictor variables into X dataframe. 'Final_grade' is dropped as it is dependent variable
Ymind = dmind['treatment']# Copy the 'Final_grade' column alone into the y dataframe. This is the dependent variable 
seed=25
#now we break the X and y dataframes into training set and test set. For this we will use
#Sklearn package's data splitting function which is based on random function.
Xmind_train, Xmind_test, Ymind_train, Ymind_test = train_test_split(Xmind, Ymind, test_size=0.2, random_state=25,stratify=Ymind)# Splitting X and y into training and test set in 80:20 ratio

In [36]:
dmind.dtypes

Gender                       int64
Occupation                   int64
self_employed              float64
family_history               int64
treatment                    int64
Days_Indoors                 int64
Growing_Stress             float64
Changes_Habits             float64
Mental_Health_History      float64
Mood_Swings                  int64
Coping_Struggles             int64
Work_Interest              float64
Social_Weakness            float64
mental_health_interview    float64
care_options               float64
dtype: object

In [37]:
# Import the Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest model with a fixed random state for reproducibility
rf_model_mental = RandomForestClassifier(
    max_depth=5,          # limits how deep each tree grows
    min_samples_split=10, # needs 10 samples before splitting
    min_samples_leaf=4,   # each leaf needs at least 4 samples
    n_estimators=100,
    class_weight='balanced',
    random_state=25
)

# Train the model on the training data
rf_model_mental.fit(Xmind_train, Ymind_train)

# Predict the target values for the test data
rf_pred_mental = rf_model_mental.predict(Xmind_test)

In [38]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(Ymind_test, rf_pred_mental))# Calculating the accuracy score of the model
print(classification_report(Ymind_test, rf_pred_mental))
print("CONFUSION MATRIX:",metrics.confusion_matrix(Ymind_test, rf_pred_mental))#Printing the confusion matrix to evaluate prediction results

Accuracy: 0.7129786397140561
              precision    recall  f1-score   support

           0       0.73      0.66      0.69     28952
           1       0.70      0.77      0.73     29521

    accuracy                           0.71     58473
   macro avg       0.72      0.71      0.71     58473
weighted avg       0.71      0.71      0.71     58473

CONFUSION MATRIX: [[19102  9850]
 [ 6933 22588]]


In [39]:
import joblib
import os

os.makedirs("models", exist_ok=True)

# Save physical model
joblib.dump(rf_model_body,   "models/physical_model.pkl")

# Save mental model
joblib.dump(rf_model_mental, "models/mental_model.pkl")

print("✅ Both models saved successfully!")

✅ Both models saved successfully!


In [40]:
from recommend import get_combined_health_index

score, status, color, emoji = get_combined_health_index(0.65, 0.72)
print(f"{emoji} Combined Risk: {score}% — {status}")

🟡 Combined Risk: 68.5% — Needs Some Attention


In [41]:
print(rf_model_mental.feature_names_in_)

['Gender' 'Occupation' 'self_employed' 'family_history' 'Days_Indoors'
 'Growing_Stress' 'Changes_Habits' 'Mental_Health_History' 'Mood_Swings'
 'Coping_Struggles' 'Work_Interest' 'Social_Weakness'
 'mental_health_interview' 'care_options']


In [42]:
print(dmind['Occupation'].unique())
print(dmind['Days_Indoors'].unique())
print(dmind['Mood_Swings'].unique())

[0 1 2 3 4]
[0 1 2 3 4]
[0 1 2]
